# Module `algorithms/dynamic_runner.py`

`execute_dynamic` orchestre la resolution **dynamique** : un seul vehicule execute les sous-tournees en sequence, et entre chaque transition le simulateur met a jour les couts / statuts des aretes.

Trois strategies supportees via le parametre `strategy` :

- `"fixed"` : plan initial suivi jusqu'au bout, aucune reaction aux evolutions de couts.
- `"reopt"` : re-resolution **complete** par le solveur (GRASP / SA / tabou / GA) a chaque retour depot. Cout CPU eleve, surtout pour le GA.
- `"reactive"` : plan initial calcule une seule fois au step 0, puis aucun appel au solveur lourd. A chaque transition, une passe de **2-opt** reordonne le suffixe de la sous-tournee courante avec la matrice de couts du snapshot ; au retour depot, `local_search` (relocate + swap inter-routes + 2-opt) rebat le plan restant. C'est l'option qui donne le meilleur compromis qualite / temps.

Le parametre historique `reoptimize_at_depot` reste accepte (il choisit entre `fixed` et `reopt` quand `strategy` n'est pas passe).

`DynamicExecution` contient la trace : route parcourue, couts pas a pas, nombre d'appels solveur (`reoptimizations`), compteurs detailles par type (`depot_replans`, `reactive_repairs`) et temps cumule.


In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent / "src"))

from cesipath import GraphGenerationConfig, GraphGenerator
from cesipath.dynamic_network import DynamicNetworkSimulator
from cesipath.algorithms import execute_dynamic, grasp


## 1. Mise en place

On construit :

1. une `GraphInstance` (statique),
2. un `DynamicNetworkSimulator` (qui fait evoluer les couts),
3. on passe tout a `execute_dynamic` avec un solveur de base (ici GRASP).


In [ ]:
cfg = GraphGenerationConfig(node_count=12, seed=3)
instance = GraphGenerator(cfg).generate()
simulator = DynamicNetworkSimulator(instance, seed=0)

exec_result = execute_dynamic(
    instance,
    simulator,
    grasp,
    solver_kwargs={"max_iterations": 15, "rcl_alpha": 0.3, "seed": 0},
    strategy="reactive",
)

print(f"plan initial        : {exec_result.planned_cost:.2f}")
print(f"cout realise        : {exec_result.realized_cost:.2f}")
print(f"appels solveur      : {exec_result.reoptimizations}")
print(f"depot replans       : {exec_result.depot_replans}")
print(f"reactive repairs    : {exec_result.reactive_repairs}")
print(f"temps solveur cumule: {exec_result.solver_time:.3f} s")
print(f"nb de pas           : {exec_result.total_steps}")


## 2. Lecture de la trace

`traveled_route` est la sequence reelle de noeuds visites (y compris les passages au depot). `step_costs` est la liste des couts dynamiques effectivement payes a chaque transition.


In [3]:
print("route parcourue :")
print(" ", exec_result.traveled_route)
print()
print("couts pas a pas :")
for i, c in enumerate(exec_result.step_costs):
    u = exec_result.traveled_route[i]
    v = exec_result.traveled_route[i + 1]
    print(f"  step {i + 1:>2} : {u} -> {v}  cout={c:.2f}")


route parcourue :
  [0, 8, 7, 11, 5, 6, 9, 1, 2, 0, 3, 4, 10, 0]

couts pas a pas :
  step  1 : 0 -> 8  cout=51.22
  step  2 : 8 -> 7  cout=79.15
  step  3 : 7 -> 11  cout=50.37
  step  4 : 11 -> 5  cout=34.96
  step  5 : 5 -> 6  cout=17.38
  step  6 : 6 -> 9  cout=41.03
  step  7 : 9 -> 1  cout=27.69
  step  8 : 1 -> 2  cout=61.62
  step  9 : 2 -> 0  cout=69.86
  step 10 : 0 -> 3  cout=42.98
  step 11 : 3 -> 4  cout=74.20
  step 12 : 4 -> 10  cout=163.96
  step 13 : 10 -> 0  cout=66.56


## 3. Comparer les trois strategies

On fait tourner `fixed`, `reopt` et `reactive` sur la meme seed du simulateur pour une comparaison equitable. `reactive` est celle utilisee en pratique : meme plan initial que les autres, mais zero appel solveur lourd apres l'init.


In [ ]:
def run(strategy: str):
    sim = DynamicNetworkSimulator(instance, seed=0)
    return execute_dynamic(
        instance, sim, grasp,
        solver_kwargs={"max_iterations": 15, "rcl_alpha": 0.3, "seed": 0},
        strategy=strategy,
    )

runs = {s: run(s) for s in ("fixed", "reopt", "reactive")}

header = f"{'strategy':<10} {'realise':>8} {'solveur':>7} {'depot':>5} {'react':>5} {'temps':>7}"
print(header)
for name, r in runs.items():
    print(f"{name:<10} {r.realized_cost:>8.2f} {r.reoptimizations:>7d} {r.depot_replans:>5d} {r.reactive_repairs:>5d} {r.solver_time:>6.3f}s")

fixed = runs["fixed"].realized_cost
for name in ("reopt", "reactive"):
    gain = 100 * (fixed - runs[name].realized_cost) / fixed
    print(f"gain {name:<8} vs fixed : {gain:+.1f} %")


## 4. Convention des demandes restantes

Lors d'une re-optimisation `reopt`, `_restricted_solver_input` construit un `SolverInput` avec les demandes des clients deja servis mises **a zero**. Les solveurs filtrent les clients de demande nulle, ce qui les exclut sans alterer la matrice de couts (important pour conserver l'inegalite triangulaire qui passe par Δ-TSP). La strategie `reactive` n'en a pas besoin : elle travaille directement sur les routes restantes via `local_search`.


## 5. Cout planifie vs cout realise

Le `planned_cost` est le cout que l'algo a calcule sur le snapshot **initial**. Le `realized_cost` integre les evolutions des couts entre chaque pas. La difference `realized - planned` mesure l'impact de la dynamique sur la performance apparente.


In [5]:
print(f"planifie : {exec_result.planned_cost:.2f}")
print(f"realise  : {exec_result.realized_cost:.2f}")
print(f"ecart    : {exec_result.realized_cost - exec_result.planned_cost:+.2f}")


planifie : 608.50
realise  : 780.98
ecart    : +172.48


## 6. Interchangeabilite des solveurs

`execute_dynamic` accepte n'importe quelle fonction de signature `SolverInput -> VRPSolution` : GRASP, SA, tabou, GA. Avec la strategie `reactive` le solveur n'est appele **qu'une seule fois** (plan initial), ce qui rend le temps total pratiquement independant du choix d'algo apres l'init — le GA notamment passe de O(depot_replans x cout_GA) en `reopt` a O(cout_GA) en `reactive`.

C'est exactement le role de `dynamic_benchmark.py` : mesurer le compromis qualite / temps des trois strategies sur les quatre metaheuristiques.
